# DocFinder — Indexeur Colab (GPU)

Ce notebook tourne sur Colab GPU et prend en charge tout le travail lourd d'embedding.

## Architecture
```
Mac (local) ──── /chunks NDJSON ───► Colab GPU ──── embeddings ───► Mac /admin/upsert ───► Qdrant local
              (extraction texte)    (sentence-transformers)         (FastAPI port 8000)    (port 6333)
```

Colab ne se connecte **jamais** directement à Qdrant — tout passe par le serveur FastAPI du Mac via ngrok.

## Flux automatique
1. Colab poll `/admin/ping` toutes les 5s via ngrok
2. Quand `status=running` → fetch `/chunks` en streaming
3. Calcule les embeddings dense sur GPU (FP16 si CUDA)
4. POST les points vers `/admin/upsert` sur le Mac → le Mac écrit dans Qdrant local

## Prérequis
- Serveur FastAPI en cours d'exécution sur le Mac : `bash start.sh`
- **Un seul tunnel ngrok** suffit : `ngrok http 8000` (port 8000 uniquement)

## Démarrage
1. Renseignez `DOCFINDER_NGROK_URL` dans la cellule **Config** ci-dessous
2. **Runtime → Run all** — le daemon démarre automatiquement à la dernière cellule
3. Cliquez **Lancer l'indexation** dans http://localhost:8000/admin

In [ ]:
# ── Installation des dépendances ──────────────────────────────────────────────
# sentence-transformers charge le modèle paraphrase-multilingual-mpnet-base-v2
# requests gère les appels HTTP vers le Mac via ngrok
!pip install -q sentence-transformers requests

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Config — seule cellule à modifier
# ─────────────────────────────────────────────────────────────────────────────

# URL ngrok du Mac (port 8000) — à mettre à jour à chaque restart de ngrok
# Sur le Mac : ngrok http 8000  → copiez l'URL HTTPS affichée
DOCFINDER_NGROK_URL = "https://CHANGE-ME.ngrok-free.app"

# Modèle d'embedding (doit être identique au serveur local)
MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"

# Nombre de points accumulés avant chaque envoi au Mac
# Plus la valeur est grande, moins il y a de round-trips ngrok
UPSERT_EVERY = 512

# Intervalle de polling du daemon (secondes)
POLL_INTERVAL = 5

# ── Validation immédiate ──────────────────────────────────────────────────────
if "CHANGE-ME" in DOCFINDER_NGROK_URL or not DOCFINDER_NGROK_URL.startswith("https://"):
    raise ValueError(
        "\u26d4  Mettez à jour DOCFINDER_NGROK_URL avant de continuer.\n"
        "    Sur le Mac : ngrok http 8000 \u2192 copiez l'URL HTTPS affichée\n"
        "    Exemple : https://abc123.ngrok-free.app"
    )

print(f"DocFinder URL : {DOCFINDER_NGROK_URL}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Setup : GPU, modèle, connexion Mac
# ─────────────────────────────────────────────────────────────────────────────
import time
import requests
import torch
from sentence_transformers import SentenceTransformer

NGROK_HEADERS = {"ngrok-skip-browser-warning": "true"}

# ── Détection GPU et réglage automatique du BATCH_SIZE ───────────────────────
# Le BATCH_SIZE contrôle combien de chunks sont encodés simultanément.
# Valeurs choisies pour occuper ~80% de la VRAM sans OOM sur les modèles courants.
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

    if vram_gb >= 75:      # A100 80 GB
        BATCH_SIZE = 256
    elif vram_gb >= 20:    # L4 (22.5 GB), A10G (23.7 GB)
        BATCH_SIZE = 128
    elif vram_gb >= 14:    # T4 15.8 GB (Colab gratuit)
        BATCH_SIZE = 64
    else:                  # GPU modeste (< 14 GB)
        BATCH_SIZE = 32

    device = "cuda"
    print(f"GPU : {gpu_name} ({vram_gb:.1f} GB VRAM) \u2192 BATCH_SIZE={BATCH_SIZE}")
else:
    device     = "cpu"
    BATCH_SIZE = 16
    gpu_name   = "CPU"
    print(f"Aucun GPU \u2014 CPU seul, BATCH_SIZE={BATCH_SIZE}")

# ── Chargement du modèle ──────────────────────────────────────────────────────
model = SentenceTransformer(MODEL_NAME, device=device)

if device == "cuda":
    # FP16 : réduit la VRAM de moitié et accélère l'encodage d'environ 2x sur T4/A100.
    # La perte de précision est négligeable pour la similarité cosinus.
    model.half()
    print(f"FP16 activé \u2014 encodage ~2\u00d7 plus rapide sur {gpu_name}")

print(f"Mod\u00e8le '{MODEL_NAME}' pr\u00eat.")

# ── Test de connexion au Mac ──────────────────────────────────────────────────
try:
    r = requests.get(f"{DOCFINDER_NGROK_URL}/health", headers=NGROK_HEADERS, timeout=15)
    r.raise_for_status()
    health = r.json()
    assert health.get("status") == "ok", f"R\u00e9ponse inattendue : {health}"
    print(f"Connexion Mac OK \u2713  ({health})")
except requests.exceptions.ConnectionError:
    raise SystemError(
        "\u26d4  Impossible de joindre le Mac.\n"
        "    V\u00e9rifiez que :\n"
        "    1. Le serveur FastAPI tourne (bash start.sh sur le Mac)\n"
        "    2. ngrok tourne sur le Mac (ngrok http 8000)\n"
        "    3. DOCFINDER_NGROK_URL correspond bien \u00e0 l'URL ngrok actuelle"
    )
except requests.exceptions.HTTPError as exc:
    raise SystemError(
        f"\u26d4  Le Mac a r\u00e9pondu avec une erreur HTTP : {exc}\n"
        "    V\u00e9rifiez que le serveur FastAPI est bien d\u00e9marr\u00e9."
    )

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Helpers réseau : retry HTTP, heartbeat, control, stream producteur
# ─────────────────────────────────────────────────────────────────────────────
import json
import queue
import threading

# Délais du backoff exponentiel (secondes) — 5 tentatives, cap à 16s.
# Couvre les coupures ngrok transitoires (idle timeout, tunnel restart).
_RETRY_DELAYS = (1, 2, 4, 8, 16)

# Nombre max de batches en avance dans la queue producteur → consommateur.
# Limite la consommation mémoire si l'encodage GPU est plus rapide que le réseau.
QUEUE_MAXSIZE = 4


def _http_get(url: str, **kwargs) -> requests.Response:
    """GET avec retry exponentiel sur ConnectionError, Timeout et 502/503/504.

    ngrok peut couper la connexion (tunnel restart, idle timeout). Ce helper
    réessaie automatiquement jusqu'à 5 fois avec délai croissant (1→2→4→8→16s).
    """
    for attempt, delay in enumerate(_RETRY_DELAYS, 1):
        try:
            r = requests.get(url, headers=NGROK_HEADERS, **kwargs)
            if r.status_code in (502, 503, 504):
                raise requests.exceptions.ConnectionError(f"HTTP {r.status_code}")
            return r
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
            if attempt == len(_RETRY_DELAYS):
                raise
            print(f"  [retry {attempt}/{len(_RETRY_DELAYS)}] {exc} \u2192 dans {delay}s")
            time.sleep(delay)
    raise RuntimeError("unreachable")


def _http_post(url: str, **kwargs) -> requests.Response:
    """POST avec retry exponentiel — même logique que _http_get."""
    for attempt, delay in enumerate(_RETRY_DELAYS, 1):
        try:
            r = requests.post(url, headers=NGROK_HEADERS, **kwargs)
            if r.status_code in (502, 503, 504):
                raise requests.exceptions.ConnectionError(f"HTTP {r.status_code}")
            return r
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
            if attempt == len(_RETRY_DELAYS):
                raise
            print(f"  [retry {attempt}/{len(_RETRY_DELAYS)}] {exc} \u2192 dans {delay}s")
            time.sleep(delay)
    raise RuntimeError("unreachable")


def get_status() -> dict:
    """Interroge /admin/ping sur le Mac et retourne le statut du job courant."""
    r = _http_get(f"{DOCFINDER_NGROK_URL}/admin/ping", timeout=30)
    return r.json()


def send_heartbeat() -> None:
    """Envoie un heartbeat au Mac (GPU name + device) — non-bloquant.

    Permet à l'admin UI d'afficher l'état de connexion Colab.
    Les erreurs sont silencieusement ignorées pour ne pas interrompre le daemon.
    """
    try:
        _http_post(
            f"{DOCFINDER_NGROK_URL}/admin/colab/heartbeat",
            json={"device": device, "gpu": gpu_name},
            timeout=5,
        )
    except Exception:
        pass


def check_paused() -> bool:
    """Retourne True si le Mac a demandé une pause via le bouton Pause de l'admin UI."""
    try:
        r = _http_get(f"{DOCFINDER_NGROK_URL}/admin/colab/control", timeout=5)
        return r.json().get("paused", False)
    except Exception:
        return False


def fetch_indexed_doc_ids() -> set:
    """Récupère les doc_id déjà présents dans Qdrant pour permettre la reprise.

    En cas d'interruption (kernel crash, réseau coupé), les documents déjà
    indexés sont ignorés lors du prochain lancement. Si l'endpoint échoue
    (ancienne version serveur, réseau instable), retourne un ensemble vide
    et l'indexation repart de zéro.
    """
    try:
        r = _http_get(f"{DOCFINDER_NGROK_URL}/admin/indexed-doc-ids", timeout=30)
        ids = set(r.json().get("doc_ids", []))
        if ids:
            print(f"  Reprise : {len(ids)} docs d\u00e9j\u00e0 index\u00e9s \u2014 ignor\u00e9s.")
        return ids
    except Exception as exc:
        print(f"  Avertissement : impossible de r\u00e9cup\u00e9rer les doc_ids existants ({exc})")
        return set()


def _stream_producer(path, batch_q: queue.Queue, skip_doc_ids: set) -> None:
    """Thread producteur : consomme le flux NDJSON /chunks et remplit la queue.

    Découpe le flux en batches de BATCH_SIZE chunks. Dépose None en fin
    de queue (sentinel) pour signaler la fin au consommateur principal.
    Les chunks des docs déjà indexés (skip_doc_ids) sont silencieusement ignorés.
    """
    url = f"{DOCFINDER_NGROK_URL}/chunks"
    if path:
        url += f"?path={requests.utils.quote(path)}"
    batch = []
    skipped = 0
    try:
        with requests.get(url, headers=NGROK_HEADERS, stream=True, timeout=300) as resp:
            resp.raise_for_status()
            for raw_line in resp.iter_lines():
                if not raw_line:
                    continue
                line = json.loads(raw_line)
                t = line.get("type")
                if t == "meta":
                    print(f"  {line['total_files']} fichiers \u00e0 traiter")
                elif t == "file":
                    print(f"  {line['path']} \u2192 {line['chunks']} chunks")
                elif t == "skip":
                    print(f"  (ignor\u00e9) {line['path']}")
                elif t == "chunk":
                    if line.get("doc_id") in skip_doc_ids:
                        skipped += 1
                        continue
                    batch.append(line)
                    if len(batch) >= BATCH_SIZE:
                        batch_q.put(batch)
                        batch = []
                elif t == "done":
                    msg = "  Flux termin\u00e9."
                    if skipped:
                        msg += f" ({skipped} chunks ignor\u00e9s \u2014 d\u00e9j\u00e0 index\u00e9s)"
                    print(msg)
                elif t == "error":
                    print(f"  ERREUR flux : {line.get('error')}")
                    break
        if batch:
            batch_q.put(batch)
    except Exception as exc:
        print(f"  ERREUR producteur : {exc}")
    finally:
        batch_q.put(None)  # sentinel de fin


def _upsert_batch(points_data: list) -> None:
    """Envoie les points encodés au Mac via /admin/upsert (avec retry HTTP).

    Le Mac reçoit les vecteurs (dense + sparse) et les écrit dans Qdrant local.
    Utilise _http_post pour absorber les coupures ngrok automatiquement.
    """
    r = _http_post(
        f"{DOCFINDER_NGROK_URL}/admin/upsert",
        json=points_data,
        timeout=120,
    )
    r.raise_for_status()
    resp = r.json()
    if "error" in resp:
        raise RuntimeError(f"Upsert refus\u00e9 par le Mac : {resp['error']}")


print("Helpers r\u00e9seau pr\u00eats.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Moteur d'indexation GPU : _encode_safe + index_pipeline
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np


def _encode_safe(texts: list) -> tuple:
    """Encode les textes sur GPU avec gestion automatique des OOM CUDA.

    En cas de OutOfMemoryError, réduit de moitié le batch_size d'encodage
    et réessaie jusqu'à ce que ça passe ou que batch_size atteigne 1.
    Met à jour BATCH_SIZE globalement pour éviter de retomber en OOM
    sur les batches futurs du même job.

    Retourne (embeddings_array, batch_size_effectif).
    """
    global BATCH_SIZE
    bs = BATCH_SIZE
    while True:
        try:
            embs = model.encode(
                texts,
                batch_size=bs,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            return embs, bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            bs = max(1, bs // 2)
            BATCH_SIZE = bs   # persiste pour les batches suivants
            print(f"  \u26a0 OOM CUDA \u2014 BATCH_SIZE r\u00e9duit \u00e0 {bs}, retry\u2026")


def index_pipeline(path=None) -> int:
    """Pipeline GPU producer/consumer pour l'indexation de documents.

    Architecture :
    - Thread producteur (_stream_producer) : stream NDJSON /chunks → queue de batches
    - Thread principal (consommateur)      : encode sur GPU → accumule → upsert Mac

    Fonctionnalités :
    - Reprise automatique : ignore les docs déjà indexés dans Qdrant
    - OOM CUDA : réduit automatiquement le batch_size et réessaie (_encode_safe)
    - FP16 : si activé en cellule Setup, model.encode() reste FP16 tout au long
    - Pause/reprise : interroge /admin/colab/control entre chaque batch
    - Flush accumulé : envoie UPSERT_EVERY points par appel HTTP (réduit les round-trips)
    - Libération mémoire GPU après chaque flush (torch.cuda.empty_cache)

    Retourne le nombre total de points insérés dans Qdrant.
    """
    skip_doc_ids = fetch_indexed_doc_ids()

    batch_q = queue.Queue(maxsize=QUEUE_MAXSIZE)
    producer = threading.Thread(
        target=_stream_producer, args=(path, batch_q, skip_doc_ids), daemon=True
    )
    producer.start()

    total_inserted = 0
    pending_points = []

    while True:
        batch = batch_q.get()
        if batch is None:   # sentinel : le producteur a terminé
            break

        # ── Pause demandée depuis l'admin Mac ─────────────────────────────
        while check_paused():
            print("  \u23f8 En pause \u2014 en attente de reprise\u2026", end="\r")
            time.sleep(3)
        # ──────────────────────────────────────────────────────────────────

        texts = [c["content"] for c in batch]
        embeddings, _ = _encode_safe(texts)

        for chunk, emb in zip(batch, embeddings):
            p = {
                "id":      chunk["point_id"],
                "dense":   emb.tolist(),
                "payload": {
                    "doc_id":      chunk["doc_id"],
                    "chunk_id":    chunk["chunk_id"],
                    "title":       chunk["title"],
                    "path":        chunk["path"],
                    "abs_path":    chunk.get("abs_path", ""),
                    "doc_type":    chunk["doc_type"],
                    "content":     chunk["content"],
                    "keywords":    chunk["keywords"],
                    "chunk_index": chunk["chunk_index"],
                },
            }
            if chunk.get("sparse_indices"):
                p["sparse_indices"] = chunk["sparse_indices"]
                p["sparse_values"]  = chunk["sparse_values"]
            pending_points.append(p)

        del embeddings   # libérer la mémoire GPU immédiatement

        # Flush vers Mac dès qu'on a UPSERT_EVERY points
        if len(pending_points) >= UPSERT_EVERY:
            _upsert_batch(pending_points)
            total_inserted += len(pending_points)
            print(f"  \u2191 {len(pending_points)} points \u2192 Qdrant (total : {total_inserted})")
            pending_points = []
            torch.cuda.empty_cache()

    # Flush final du reste en attente
    if pending_points:
        _upsert_batch(pending_points)
        total_inserted += len(pending_points)
        print(f"  \u2191 Flush final {len(pending_points)} points \u2192 Qdrant (total : {total_inserted})")

    torch.cuda.empty_cache()
    producer.join()
    return total_inserted


print("Moteur d'indexation pr\u00eat.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Daemon — polling automatique
#
# Cette cellule démarre automatiquement lors du « Run All ».
# Elle boucle indéfiniment en attendant un job lancé depuis l'admin Mac.
#
# Arrêt : bouton ■ (interrompre le kernel) ou Ctrl+C.
# ─────────────────────────────────────────────────────────────────────────────
print("Daemon d\u00e9marr\u00e9 \u2014 en attente d'un job sur le Mac\u2026")
print(f"Poll toutes les {POLL_INTERVAL}s sur {DOCFINDER_NGROK_URL}/admin/ping")
print("Lancez une indexation depuis http://localhost:8000/admin")
print("\u2500" * 60)

job_running = False

while True:
    try:
        send_heartbeat()
        status = get_status()
        s = status.get("status")

        if s == "running" and not job_running:
            job_running = True
            print(f"\n[{time.strftime('%H:%M:%S')}] Job d\u00e9tect\u00e9 \u2014 d\u00e9marrage du pipeline\u2026")
            try:
                n = index_pipeline()
                print(f"\n[{time.strftime('%H:%M:%S')}] Termin\u00e9 \u2014 {n} points ins\u00e9r\u00e9s dans Qdrant.")
            except Exception as exc:
                print(f"[{time.strftime('%H:%M:%S')}] ERREUR pipeline : {exc}")

        elif s in ("done", "error", "idle") and job_running:
            job_running = False
            print(f"[{time.strftime('%H:%M:%S')}] Job {s}. En attente du prochain\u2026")
            print("\u2500" * 60)

        elif s == "running" and job_running:
            done  = status.get("done", 0)
            total = status.get("total", 0)
            pct   = status.get("progress_pct", 0)
            print(f"  Mac : {done}/{total} fichiers ({pct}%)\u2026", end="\r")

    except requests.exceptions.ConnectionError:
        print(f"[{time.strftime('%H:%M:%S')}] Mac inaccessible \u2014 retry dans {POLL_INTERVAL}s\u2026")
    except Exception as exc:
        print(f"[{time.strftime('%H:%M:%S')}] Erreur : {exc}")

    time.sleep(POLL_INTERVAL)